# Model Comparison

Đọc các `metrics.json` trong `experiments/`, sau đó xếp hạng bốn model regression theo RMSE. Logistic Regression được hiển thị riêng vì đó là classification.

In [6]:
from pathlib import Path
import json
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
experiment_root = PROJECT_ROOT / 'experiments'
model_tasks = {
    'gradient_boosting': 'regression',
    'svr': 'regression',
    'mlp': 'regression',
    'pytorch_mlp': 'regression',
    'logistic_regression': 'classification',
}

# Use the newest run for each model so old experiment formats do not create duplicates.
records = []
model_summaries = []
for model_name, task in model_tasks.items():
    model_root = experiment_root / model_name
    metrics_paths = sorted(model_root.glob('*/metrics.json'))
    if not metrics_paths:
        continue

    metrics_path = metrics_paths[-1]
    run_dir = metrics_path.parent
    metrics = json.loads(metrics_path.read_text(encoding='utf-8'))
    config = json.loads((run_dir / 'config.json').read_text(encoding='utf-8'))
    for metric, value in metrics.items():
        records.append({
            'model': model_name,
            'task': task,
            'run_id': run_dir.name,
            'metric': metric,
            'value': value,
        })
    model_summaries.append({
        'model': model_name,
        'task': task,
        'run_id': run_dir.name,
        'parameters': config.get('best_params', config),
        **metrics,
    })

comparison = pd.DataFrame(records)
summary = pd.DataFrame(model_summaries)
if comparison.empty:
    print('Chưa có kết quả experiment. Hãy chạy các notebook huấn luyện trước.')
else:
    print('Tham số và validation metrics của tất cả 5 mô hình')
    display(summary)

    regression = comparison[comparison['task'] == 'regression']
    if not regression.empty:
        print('Xếp hạng regression theo RMSE (thấp hơn tốt hơn)')
        regression_ranking = (
            regression[regression['metric'] == 'rmse']
            .sort_values('value')
            .reset_index(drop=True)
        )
        display(regression_ranking)
        best_regression = regression_ranking.iloc[0]
        print(f"Best regression model: {best_regression['model']} | RMSE: {best_regression['value']:.4f}")

    classification = comparison[comparison['task'] == 'classification']
    if not classification.empty:
        print('Validation metrics của Logistic Regression')
        display(classification.sort_values('metric').reset_index(drop=True))
        best_classification = classification[classification['metric'] == 'roc_auc'].sort_values('value', ascending=False).iloc[0]
        print(f"Best classification model: {best_classification['model']} | ROC-AUC: {best_classification['value']:.4f}")

    comparison.to_csv(experiment_root / 'model_comparison.csv', index=False)
    summary.to_csv(experiment_root / 'model_summary.csv', index=False)

Tham số và validation metrics của tất cả 5 mô hình


,model,task,run_id,parameters,mae,rmse,r2,rmsle,roc_auc,accuracy
0,gradient_boosting,regression,20260919_180037,"{'regressor__learning_rate': 0.1, 'regressor__...",15932.272119,26367.221158,0.909361,0.136436,NaN,NaN
1,svr,regression,20260919_180104,"{'regressor__C': 300, 'regressor__epsilon': 0....",45997.121270,77427.487288,0.218414,0.353492,NaN,NaN
2,mlp,regression,20260919_180152,"{'regressor__alpha': 0.001, 'regressor__hidden...",21396.918262,37227.459205,0.819319,0.163942,NaN,NaN
3,pytorch_mlp,regression,20260919_180218,"{'model': 'PyTorch MLP', 'task': 'regression',...",16385.035156,28057.849953,0.897365,0.135407,NaN,NaN
4,logistic_regression,classification,20260919_180004,"{'classifier__C': 0.01, 'classifier__class_wei...",NaN,NaN,NaN,NaN,0.979874,0.921233


Xếp hạng regression theo RMSE (thấp hơn tốt hơn)


,model,task,run_id,metric,value
0,gradient_boosting,regression,20260919_180037,rmse,26367.221158
1,pytorch_mlp,regression,20260919_180218,rmse,28057.849953
2,mlp,regression,20260919_180152,rmse,37227.459205
3,svr,regression,20260919_180104,rmse,77427.487288


Best regression model: gradient_boosting | RMSE: 26367.2212
Validation metrics của Logistic Regression


,model,task,run_id,metric,value
0,logistic_regression,classification,20260919_180004,accuracy,0.921233
1,logistic_regression,classification,20260919_180004,roc_auc,0.979874


Best classification model: logistic_regression | ROC-AUC: 0.9799
